# Computing the Singular Value Decomposition (SVD) of a matrix using QR iteation
We saw how to compute the eigendecomposition of a matrix using the QR iteration method. We can use a similar approach to compute the Singular Value Decomposition (SVD) of a matrix. Let's dig into these details.

The singular value decomposition _factors any matrix_ into three components. 
Given a matrix $\mathbf{A}\in\mathbb{R}^{m\times n}$ of rank $r$, the **full** SVD is given by the factorization:
$$
\boxed{
\mathbf{A} = \mathbf{U}\,\mathbf{S}\,\mathbf{V}^\top
}
$$

where:
* $\mathbf{U}\in\mathbb{R}^{m\times m}$ is orthogonal ($\mathbf{U}^\top \mathbf{U} = \mathbf{I}_m$),
* $\mathbf{S}\in\mathbb{R}^{m\times n}$ is rectangular diagonal (with the singular values $\sigma_1\ge\cdots\ge\sigma_r>0$ on the first $r$ diagonal entries, zeros elsewhere). The singular values are the square roots of the non-negative eigenvalues of the positive semi-definite matrix $\mathbf{A}^{\top}\mathbf{A}$, i.e., $\sigma_i = \sqrt{\lambda_i(\mathbf{A}^\top\mathbf{A})}$, 
where $\lambda_i(\mathbf{A}^\top\mathbf{A})\geq 0$ denote the eigenvalues of $\mathbf{A}^\top\mathbf{A}$. The number of non-zero singular values is [the rank](https://en.wikipedia.org/wiki/Rank_(linear_algebra)) of the matrix $\mathbf{A}$, where $r \leq\min\left(n,m\right)$.
* $\mathbf{V}\in\mathbb{R}^{n\times n}$ is orthogonal ($\mathbf{V}^\top \mathbf{V} = \mathbf{I}_n$).

So what is the connection with the eigendecomposition? Let's see
___

## Naive approach: Eigendecomposition of $\mathbf{A}^\top \mathbf{A}$
The most straightforward way to compute the SVD of a matrix $\mathbf{A}$ is to compute the eigendecomposition of the matrix $\mathbf{A}^\top \mathbf{A}$, which is symmetric and positive semi-definite.

Because $\mathbf{A}^\top\mathbf{A}$ is symmetric and positive semidefinite, it has an orthogonal eigendecomposition:
$$
\mathbf{A}^\top\mathbf{A} = \mathbf{V}\mathbf{\Lambda}\mathbf{V}^\top
$$
where $\mathbf{\Lambda}=\mathrm{diag}(\lambda_1,\dots,\lambda_n)$ with $\lambda_i\geq 0$ and $\mathbf{V}=[\mathbf{v}_1,\dots,\mathbf{v}_n]$.

For any eigenpair $(\lambda_i,\mathbf{v}_i)$ with $\lambda_i>0$, define:
$$
\sigma_i = \sqrt{\lambda_i},\qquad \mathbf{u}_i = \frac{1}{\sigma_i}\mathbf{A}\mathbf{v}_i
$$
Then
$$
\mathbf{A}\mathbf{v}_i = \sigma_i \mathbf{u}_i
$$
and the vectors $\mathbf{u}_i$ are orthonormal:
$$
\mathbf{u}_i^\top\mathbf{u}_j = \frac{1}{\sigma_i\sigma_j}\mathbf{v}_i^\top\mathbf{A}^\top\mathbf{A}\mathbf{v}_j
= \frac{\lambda_j}{\sigma_i\sigma_j}\mathbf{v}_i^\top\mathbf{v}_j = \delta_{ij}.
$$
If $\lambda_i=0$, then $\mathbf{A}\mathbf{v}_i=\mathbf{0}$, so $\mathbf{v}_i$ lies in the nullspace of $\mathbf{A}$. Collecting the nonzero singular values gives the reduced SVD:
$$
\mathbf{A} = \mathbf{U}_r \mathbf{S}_r \mathbf{V}_r^\top,
$$
where $\mathbf{S}_r=\mathrm{diag}(\sigma_1,\dots,\sigma_r)$. A full $\mathbf{U}$ can be obtained by completing $\mathbf{U}_r$ with an orthonormal basis for the left nullspace of $\mathbf{A}$.

#### Algorithm: Naive SVD via eigendecomposition of $\mathbf{A}^\top\mathbf{A}$

__Initialization__: Given $\mathbf{A}\in\mathbb{R}^{m\times n}$, form $\mathbf{B}\gets\mathbf{A}^\top\mathbf{A}$ and choose a nonnegative tolerance $\tau$ for clipping tiny negative eigenvalues (e.g., $\tau\approx 0$ or a small multiple of machine precision).

> __Numerical note__: In floating-point arithmetic, $\mathbf{B}$ can have small negative eigenvalues due to roundoff. Use $\max(\lambda_i,0)$ (or $\max(\lambda_i,\tau)$) before taking square roots.

Compute the eigendecomposition $\mathbf{B} = \mathbf{V}\mathbf{\Lambda}\mathbf{V}^\top$ (e.g., using QR iteration for symmetric matrices).

Sort the eigenvalues $\lambda_i$ in descending order and reorder the columns of $\mathbf{V}$ accordingly.

Define singular values $\sigma_i \gets \sqrt{\max(\lambda_i,0)}$ and construct the rectangular diagonal matrix $\mathbf{S}$.

For each $\sigma_i>0$ __do__:
- Compute $\mathbf{u}_i \gets \mathbf{A}\mathbf{v}_i/\sigma_i$.
- Assemble $\mathbf{U}_r=[\mathbf{u}_1,\dots,\mathbf{u}_r]$.

If a full $\mathbf{U}$ is required, extend $\mathbf{U}_r$ with an orthonormal basis for the left nullspace of $\mathbf{A}$.

__Output__: $\mathbf{A}=\mathbf{U}\mathbf{S}\mathbf{V}^\top$.

## Summary: Eigendecomposition-based SVD

We have established the relationship between the eigendecomposition of $\mathbf{A}^\top\mathbf{A}$ and the SVD of $\mathbf{A}$:

**Key Results:**
1. The eigenvalues of $\mathbf{A}^\top\mathbf{A}$ are nonnegative and satisfy $\lambda_i=\sigma_i^2$.
2. The right singular vectors $\mathbf{V}$ are the eigenvectors of $\mathbf{A}^\top\mathbf{A}$.
3. For $\lambda_i>0$, the left singular vectors are $\mathbf{u}_i=\mathbf{A}\mathbf{v}_i/\sigma_i$, yielding $\mathbf{A}=\mathbf{U}_r\mathbf{S}_r\mathbf{V}_r^\top$.

**Important Observations:**
- Forming $\mathbf{A}^\top\mathbf{A}$ squares the condition number: $\kappa(\mathbf{A}^\top\mathbf{A})=\kappa(\mathbf{A})^2$, so small singular values are easily lost.
- Roundoff can produce small negative eigenvalues that must be clipped before taking square roots.
- Forming $\mathbf{A}^\top\mathbf{A}$ can be expensive and can destroy sparsity; it also requires two passes over $\mathbf{A}$.
- Practical SVD algorithms avoid explicitly forming $\mathbf{A}^\top\mathbf{A}$ (e.g., bidiagonalization followed by QR).

___
